In [1]:
# pip install tensorflow==2.15.0

In [2]:
import tensorflow as tf

print("tf.__version__ =", tf.__version__)


tf.__version__ = 2.15.0


### 샘플 예제

In [3]:
import numpy as np

def sigmoid(x):   # 시그모이드 함수
    return 1. / (1 + np.exp(-x))   # 수식을 표현한 내용

In [4]:
def numerical_derivative(f, x):   # f: 손실값을 계산하는 함수 / x: 업데이트 하기 위한 변수
    delta_x = 1e-4   # 미분할 값의 크기
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index        
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x) # f(x+delta_x)
        
        x[idx] = float(tmp_val) - delta_x 
        fx2 = f(x) # f(x-delta_x)
        grad[idx] = (fx1 - fx2) / (2*delta_x)
        
        x[idx] = tmp_val 
        it.iternext()

    return grad

In [5]:
# https://f-lab.kr/insight/understanding-cross-entropy-loss-function

In [6]:
# 이게 핵심

class LogicGate:
    def __init__(self, gate_name, xdata, tdata):
        self.name = gate_name   # 게이트 이름
        self.xdata = xdata.reshape(4,2)   # 학습 데이터
        self.tdata = tdata.reshape(4,1)   # 정답 데이터

        self.W = np.random.rand(self.xdata.shape[1], 1)   # 가중치 초기값 설정
        self.b = np.random.rand(1)   # 바이어스 초기값 설정
        self.learning_rate = 1e-2    # 학습율

    def loss_func(self):
        delta = 1e-7
        z = np.dot(self.xdata, self.W) + self.b
        y = sigmoid(z)   # 활성함수: 시그모이드 함수 = 이진 분류에서 사용함
        return -np.sum(self.tdata*np.log(y+delta) + (1-self.tdata)*np.log((1-y)+delta))   # 이진 분류에 대한 손실값 / 이진 분류일 때 이 수식을 사용함

    
# 학습하는 과정을 함수로 만들 거임
    def train(self):
        f = lambda x : self.loss_func()   # 손실값 계산하는 람다 함수
        print("Initial loss value = ", self.loss_func())   # 초기 손실값 계산해서 출력
        for step in range(20000):
            self.W -= self.learning_rate * numerical_derivative(f, self.W)   # 가중치 업데이터
            self.b -= self.learning_rate * numerical_derivative(f, self.b)   # 바이어스 업데이트
            if (step % 1000 == 0):
                print("step = ", step, "loss value = ", self.loss_func())    # 헉습중 손실값 출력


# 예측하는 거 만들 거임
    def predict(self, input_data):
        z = np.dot(input_data, self.W) + self.b   # 회귀식 계산
        y = sigmoid(z)   # 활성 함수 시그모이드 함수 실행: 0~1 사이의 확률값 출력
        if y > 0.5:
            result = 1   # 분류값 저장
        else:
            result = 0
        return y, result

    
# 정확도 계산하는 거 만들 거임
    def accuracy(self, test_xdata, test_tdata):
        matched_list = []   # 정답과 일치한 내용 저장
        not_matched_list = []   # 정답과 불일치한 내용 저장
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                matched_list.append(index)   # 정답과 일치한 내용 추가
            else:
                not_matched_list.append(index)   # 정답과 불일치한 내용 추가
        accuracy_val = len(matched_list) / len(test_xdata)   # 정확도 계산
        return accuracy_val

In [7]:
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 0, 0, 1])   # 정답(target) 데이터
AND_obj = LogicGate("AND_GATE", xdata, tdata)   # 모델 객체 생성
AND_obj.train()   # 학습

Initial loss value =  3.684458351015619
step =  0 loss value =  3.6474930080581407
step =  1000 loss value =  1.0233971446811672
step =  2000 loss value =  0.6668976247654077
step =  3000 loss value =  0.4951103913928402
step =  4000 loss value =  0.39263507324811553
step =  5000 loss value =  0.3245426780869215
step =  6000 loss value =  0.2761020118339331
step =  7000 loss value =  0.23994593203399794
step =  8000 loss value =  0.21197077857525443
step =  9000 loss value =  0.18970969423947268
step =  10000 loss value =  0.17159189275083064
step =  11000 loss value =  0.15657073313950604
step =  12000 loss value =  0.14392263627749768
step =  13000 loss value =  0.13313181557278608
step =  14000 loss value =  0.12382096729038389
step =  15000 loss value =  0.11570790007900536
step =  16000 loss value =  0.10857746037894563
step =  17000 loss value =  0.10226282051083374
step =  18000 loss value =  0.09663268538968593
step =  19000 loss value =  0.09158234750967555


In [8]:
test_xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = AND_obj.predict(input_data)
    print(input_data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 0, 0, 1])
accuracy_ret = AND_obj.accuracy(test_xdata, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 0
[0 1] = 0
[1 0] = 0
[1 1] = 1
----------------------------------------
Accuracy => 1.0


In [9]:
# 바로 위에는 AND

In [10]:
# OR 확인할 거임 
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 1, 1, 1])   # 정답(target) 데이터
OR_obj = LogicGate("OR_GATE", xdata, tdata)   # 모델 객체 생성
OR_obj.train()   # 학습

Initial loss value =  1.8932522691799956
step =  0 loss value =  1.8904203868379883
step =  1000 loss value =  0.7350242353885801
step =  2000 loss value =  0.4376017660018673
step =  3000 loss value =  0.3067494432748505
step =  4000 loss value =  0.23443718870584856
step =  5000 loss value =  0.1889956365106092
step =  6000 loss value =  0.15796223165215778
step =  7000 loss value =  0.1354964038138082
step =  8000 loss value =  0.1185170179033501
step =  9000 loss value =  0.10525244310167697
step =  10000 loss value =  0.09461487461121888
step =  11000 loss value =  0.08590090283169696
step =  12000 loss value =  0.07863623194355697
step =  13000 loss value =  0.07248980409174498
step =  14000 loss value =  0.06722373562965954
step =  15000 loss value =  0.06266281632080449
step =  16000 loss value =  0.058675219220992336
step =  17000 loss value =  0.05515990237437681
step =  18000 loss value =  0.05203814799022519
step =  19000 loss value =  0.04924773945877331


In [11]:
test_xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = OR_obj.predict(input_data)
    print(input_data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 1, 1, 1])
accuracy_ret = OR_obj.accuracy(test_xdata, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 0
[0 1] = 1
[1 0] = 1
[1 1] = 1
----------------------------------------
Accuracy => 1.0


In [12]:
# NAND 확인
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([1, 1, 1, 0])   # 정답(target) 데이터
NAND_obj = LogicGate("NAND_GATE", xdata, tdata)   # 모델 객체 생성
NAND_obj.train()   # 학습

Initial loss value =  3.046510672847644
step =  0 loss value =  3.0404142024187655
step =  1000 loss value =  1.0890346667369617
step =  2000 loss value =  0.69336952640275
step =  3000 loss value =  0.509700017271455
step =  4000 loss value =  0.40188910717832504
step =  5000 loss value =  0.33092237200838803
step =  6000 loss value =  0.28075584746875726
step =  7000 loss value =  0.243484011465512
step =  8000 loss value =  0.2147472132335318
step =  9000 loss value =  0.19194389876559373
step =  10000 loss value =  0.17342689272140874
step =  11000 loss value =  0.1581036330877154
step =  12000 loss value =  0.145221609492378
step =  13000 loss value =  0.13424608809828822
step =  14000 loss value =  0.124786945211569
step =  15000 loss value =  0.11655307016801073
step =  16000 loss value =  0.10932295241579656
step =  17000 loss value =  0.10292513691952057
step =  18000 loss value =  0.09722489699126974
step =  19000 loss value =  0.09211493781917343


In [13]:
test_xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = NAND_obj.predict(input_data)
    print(input_data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([1, 1, 1, 0])
accuracy_ret = NAND_obj.accuracy(test_xdata, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 1
[0 1] = 1
[1 0] = 1
[1 1] = 0
----------------------------------------
Accuracy => 1.0


In [14]:
# XOR 확인
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 1, 1, 0])   # 정답(target) 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)   # 모델 객체 생성
XOR_obj.train()   # 학습

Initial loss value =  3.664494623923919
step =  0 loss value =  3.643928334959764
step =  1000 loss value =  2.772966434558043
step =  2000 loss value =  2.7725908512136694
step =  3000 loss value =  2.7725879587028244
step =  4000 loss value =  2.772587923204632
step =  5000 loss value =  2.7725879222771384
step =  6000 loss value =  2.772587922241429
step =  7000 loss value =  2.7725879222399286
step =  8000 loss value =  2.772587922239864
step =  9000 loss value =  2.772587922239862
step =  10000 loss value =  2.772587922239862
step =  11000 loss value =  2.7725879222398615
step =  12000 loss value =  2.7725879222398615
step =  13000 loss value =  2.772587922239861
step =  14000 loss value =  2.772587922239861
step =  15000 loss value =  2.772587922239861
step =  16000 loss value =  2.772587922239861
step =  17000 loss value =  2.772587922239861
step =  18000 loss value =  2.772587922239861
step =  19000 loss value =  2.772587922239861


In [15]:
test_xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = XOR_obj.predict(input_data)
    print(input_data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 1, 1, 0])
accuracy_ret = XOR_obj.accuracy(test_xdata, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 0
[0 1] = 0
[1 0] = 0
[1 1] = 1
----------------------------------------
Accuracy => 0.25


### (문제) NAND / OR / AND 조합을 이용한 XOR 논리 게이터 학습 및 검증 코드 작성하기

In [16]:
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 0, 0, 1])   # 정답(target) 데이터
AND_obj = LogicGate("AND_GATE", xdata, tdata)   # 모델 객체 생성
AND_obj.train()   # 학습

Initial loss value =  3.0862937547874205
step =  0 loss value =  3.0591067469622213
step =  1000 loss value =  0.9862818744462802
step =  2000 loss value =  0.6513768827745009
step =  3000 loss value =  0.48641657716432696
step =  4000 loss value =  0.38707027904874175
step =  5000 loss value =  0.32068348812761355
step =  6000 loss value =  0.27327488711694065
step =  7000 loss value =  0.23778974681976053
step =  8000 loss value =  0.21027452915443234
step =  9000 loss value =  0.18834197267570202
step =  10000 loss value =  0.17046669529868552
step =  11000 loss value =  0.1556294773936946
step =  12000 loss value =  0.14312408142433042
step =  13000 loss value =  0.13244611389175268
step =  14000 loss value =  0.12322600051545614
step =  15000 loss value =  0.11518694005733335
step =  16000 loss value =  0.10811762835247868
step =  17000 loss value =  0.10185404471943948
step =  18000 loss value =  0.09626697898220277
step =  19000 loss value =  0.09125329778285182


In [17]:
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 1, 1, 1])   # 정답(target) 데이터
OR_obj = LogicGate("AND_GATE", xdata, tdata)   # 모델 객체 생성
OR_obj.train()   # 학습

Initial loss value =  1.9938143256042786
step =  0 loss value =  1.9874841404074715
step =  1000 loss value =  0.725763477917118
step =  2000 loss value =  0.4340665818268353
step =  3000 loss value =  0.30493829524984334
step =  4000 loss value =  0.23335199370199805
step =  5000 loss value =  0.18827837713119744
step =  6000 loss value =  0.15745520546878042
step =  7000 loss value =  0.13512006107734414
step =  8000 loss value =  0.11822714657277336
step =  9000 loss value =  0.10502261137131998
step =  10000 loss value =  0.09442835466923026
step =  11000 loss value =  0.08574661347208264
step =  12000 loss value =  0.07850655239208193
step =  13000 loss value =  0.07237932696935852
step =  14000 loss value =  0.06712852112400078
step =  15000 loss value =  0.0625799279256274
step =  16000 loss value =  0.05860242468707547
step =  17000 loss value =  0.05509547529024253
step =  18000 loss value =  0.0519807326786021
step =  19000 loss value =  0.0491962568613543


In [18]:
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([1, 1, 1, 0])   # 정답(target) 데이터
NAND_obj = LogicGate("NAND_GATE", xdata, tdata)   # 모델 객체 생성
NAND_obj.train()   # 학습

Initial loss value =  3.14260774409755
step =  0 loss value =  3.1307800770873313
step =  1000 loss value =  1.0564696549135049
step =  2000 loss value =  0.6805132432599278
step =  3000 loss value =  0.5026568427993074
step =  4000 loss value =  0.39743522260899167
step =  5000 loss value =  0.3278578170388605
step =  6000 loss value =  0.27852339148633365
step =  7000 loss value =  0.24178853670242695
step =  8000 loss value =  0.21341779797646476
step =  9000 loss value =  0.19087480944494173
step =  10000 loss value =  0.17254929595133267
step =  11000 loss value =  0.15737084496085607
step =  12000 loss value =  0.14460088380925154
step =  13000 loss value =  0.133713797292381
step =  14000 loss value =  0.12432562619022369
step =  15000 loss value =  0.11614954550614592
step =  16000 loss value =  0.1089670972051752
step =  17000 loss value =  0.10260904696544315
step =  18000 loss value =  0.09694231412969068
step =  19000 loss value =  0.09186084437650191


In [19]:
input_data = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
s1 = []   # NAND 출력 저장
s2 = []   # OR 출력 저장
new_input_data = [ ]   # AND 입력 저장
final_output = [ ]     # 최종 출력 저장
for index in range(len(input_data)):   # 총 4번
    s1 = NAND_obj.predict(input_data[index])
    s2 = OR_obj.predict(input_data[index])
    new_input_data.append(s1[-1])   # 읽어와서 마지막 값을 저장
    new_input_data.append(s2[-1])
    (sigmoid_val, logical_val) = AND_obj.predict(np.array(new_input_data))
    final_output.append(logical_val)
    new_input_data = []

for index in range(len(input_data)):
    print(input_data[index], " = ", final_output[index])

[0 0]  =  0
[0 1]  =  1
[1 0]  =  1
[1 1]  =  0


### 딥러닝으로 XOR 문제 해결

In [20]:
import numpy as np

def sigmoid(x):   # 시그모이드 함수
    return 1. / (1 + np.exp(-x))   # 수식을 표현한 내용

In [21]:
def numerical_derivative(f, x):   # f: 손실값을 계산하는 함수 / x: 업데이트 하기 위한 변수
    delta_x = 1e-4   # 미분할 값의 크기
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index        
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x) # f(x+delta_x)
        
        x[idx] = float(tmp_val) - delta_x 
        fx2 = f(x) # f(x-delta_x)
        grad[idx] = (fx1 - fx2) / (2*delta_x)
        
        x[idx] = tmp_val 
        it.iternext()

    return grad

In [22]:
# 위에는 복사해서 옴 아래부터 다름

In [29]:
# 직접 이렇게 설계할 일을 없지만 이해를 돕기 위해 딥 러닝이 어떤 구조인지 공부하기 위해 만든느 것임

class LogicGate:
    def __init__(self, gate_name, xdata, tdata): # 생성자 : __init__ # 해당 클래스가 사용할 변수를 초기화할 목적으로 사용
        self.name = gate_name
        self.xdata = xdata.reshape(4,2)  # 입력층 데이터 // 입력층에 대한 설계
        self.tdata = tdata.reshape(4,1)  # 정답 데이터

        self.W2 = np.random.rand(2,6)    # 은닉층 가중치 // 은닉층에 대한 설계
        self.b2 = np.random.rand(6)      # 은닉층 바이어스

        self.W3 = np.random.rand(6,1)    # 출력증 가중치 // 출력층에 대한 설계
        self.b3 = np.random.rand(1)      # 출력층 바이어스

        self.learning_rate = 1e-2        # 학습률

    def loss_val(self):                  # 손실값 계산 
        delta = 1e-7
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2, self.W3) + self.b3
        y = a3 = sigmoid(z3)
        return -np.sum(self.tdata*np.log(y+delta)+(1-self.tdata)*np.log((1 - y)+delta)) # 최종적인 y값을 가지고 손실값 계산

# 손실함수
    def train(self):
        f = lambda x : self.loss_val() # 손실함수 정의하는 형태 
        print("Initial loss value = ", self.loss_val()) # 초기손실값 계산하는 형태
        for step in range(1001): # 2만번 학습! >> 학습을 하게 되면 각 층에 있는 가중치와 바이어스를 업데이트 시켜야한다.
            self.W2 -= self.learning_rate * numerical_derivative(f, self.W2) # 은닉층의 가중치와 바이어스를 업데이트 시킴
            self.b2 -= self.learning_rate * numerical_derivative(f, self.b2) # 은닉층의 가중치와 바이어스를 업데이트 시킴
            self.W3 -= self.learning_rate * numerical_derivative(f, self.W3) # 출력층의 가중치와 바이어스를 업데이트 시킴
            self.b3 -= self.learning_rate * numerical_derivative(f, self.b3) # 출력층의 가중치와 바이어스를 업데이트 시킴
            if (step % 1000 == 0):                                           # 중간 확인 과정
                print("step = ", step, "loss value = ", self.loss_val())

    def predict(self, input_data):
        self.xdata = input_data
        z2 = np.dot(self.xdata, self.W2) + self.b2 # 은닉층에서 연산
        a2 = sigmoid(z2)                           # 은닉층에서 연산
        z3 = np.dot(a2, self.W3) + self.b3         # 출력층에서 연산
        y = a3 = sigmoid(z3)                       # 출력층에서 연산
        if y > 0.5:
            result = 1
        else:
            result = 0

        return y, result
    
    # 정확도 계산하는 거 만들 거임
    def accuracy(self, test_xdata, test_tdata):
        matched_list = []   # 정답과 일치한 내용 저장
        not_matched_list = []   # 정답과 불일치한 내용 저장
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                matched_list.append(index)   # 정답과 일치한 내용 추가
            else:
                not_matched_list.append(index)   # 정답과 불일치한 내용 추가
        accuracy_val = len(matched_list) / len(test_xdata)   # 정확도 계산
        return accuracy_val

In [36]:
# and 게이트
# AND
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 0, 0, 1])                         # 정답(target) 데이터
AND_obj = LogicGate("AND_GATE", xdata, tdata)          # 모델 객체 생성
AND_obj.train()                                        # 학습

Initial loss value =  5.373257173569487
step =  0 loss value =  5.1646510810349096
step =  1000 loss value =  2.0436663129511348


In [37]:
test_data = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for data in test_data:
    sigmoid_val, logical_val = AND_obj.predict(data)
    print(data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 1, 1, 0])
accuracy_ret = XOR_obj.accuracy(test_xdata, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 0
[0 1] = 0
[1 0] = 0
[1 1] = 0
----------------------------------------
Accuracy => 0.5


In [34]:
# XOR 게이트
# AND
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 1, 1, 0])                         # 정답(target) 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)          # 모델 객체 생성
XOR_obj.train()                                        # 학습

Initial loss value =  6.094003913972756
step =  0 loss value =  5.960843850540305
step =  1000 loss value =  2.7704815893023085


In [35]:
test_data = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for data in test_data:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 1, 1, 0])
accuracy_ret = XOR_obj.accuracy(xdata, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 0
[0 1] = 1
[1 0] = 0
[1 1] = 1
----------------------------------------
Accuracy => 0.5


In [1]:
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation
import np_utils # pip install np_utils
from keras.utils import to_categorical
import matplotlib.pyplot as plt

In [2]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

11490434/11490434 [==============================] - 1s 0us/step


In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((60000, 28, 28), (60000,), (10000, 28, 28), (10000,))